# Clase 170 — TensorFlow.js (navegador)

Servir modelos **client-side** en el navegador con **TensorFlow.js**: privacidad (los datos no
salen del dispositivo), sin costo de servidor y sin latencia de red. Alternativas: ONNX Runtime
Web, WebGPU y transformers.js (Hugging Face) para NLP/visión en el browser.

Requiere: `numpy`; `tensorflowjs` (opcional) para la conversión. La inferencia real ocurre en JS.

## 1. Convertir un modelo Keras a formato TF.js

Con la CLI (más común):

```bash
tensorflowjs_converter --input_format=keras_saved_model servable/ tfjs_model/
# genera tfjs_model/model.json + archivos de pesos *.bin
```

In [ ]:
try:
    import tensorflowjs as tfjs
    from tensorflow import keras
    TFJS_OK = True
except Exception:
    TFJS_OK = False
    print("tensorflowjs no instalado -> conversion via CLI/API (no se ejecuta)")

if TFJS_OK:
    model = keras.models.load_model("model.keras")
    tfjs.converters.save_keras_model(model, "tfjs_model/")   # equivalente Python de la CLI
    print("modelo convertido en tfjs_model/ (model.json + pesos .bin)")
else:
    print("tfjs.converters.save_keras_model(model, 'tfjs_model/')")

## 2. Cargar el modelo e inferir desde JavaScript

```html
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
<script>
  const model = await tf.loadLayersModel('tfjs_model/model.json');
  // imagen 28x28 normalizada -> tensor (1, 784)
  const x = tf.tensor2d(pixels, [1, 784]);
  const probs = model.predict(x);
  const clase = (await probs.argMax(1).data())[0];
  console.log('clase predicha:', clase);
</script>
```

In [ ]:
import numpy as np
np.random.seed(42)

# El preprocesamiento en JS debe replicar EXACTAMENTE el de entrenamiento.
def preprocess(img28x28):
    x = img28x28.astype("float32") / 255.0     # misma normalizacion que en training
    return x.reshape(1, 784)

demo = np.random.randint(0, 256, size=(28, 28))
print("input a la red (shape):", preprocess(demo).shape)
print("mismo /255.0 y reshape hay que hacerlo en JS antes de model.predict")

## 3. Backends: WebGL, WASM y WebGPU

```js
await tf.setBackend('webgpu');   // 2-5x mas rapido que WebGL cuando esta disponible
await tf.ready();
console.log(tf.getBackend());
```

- **WebGL**: default, usa la GPU del navegador.
- **WASM**: CPU portable (fallback si no hay WebGL2).
- **WebGPU**: API moderna, la más rápida.

## 4. transformers.js: modelos de Hugging Face en el browser

```js
import { pipeline } from '@xenova/transformers';
const clf = await pipeline('sentiment-analysis');   // descarga un modelo ONNX
const out = await clf('I love this course!');
// [{ label: 'POSITIVE', score: 0.999 }]
```

Corre BERT, GPT-2, Whisper, etc. sobre ONNX Runtime Web, todo client-side.

## 5. Client-side vs server-side

| Client-side (TF.js) | Server-side |
|---|---|
| privacidad total (datos no salen) | datos viajan al server |
| sin costo de infra | costo por request/hora |
| modelo ≤ ~50 MB ideal | sin límite práctico |
| primer load descarga el modelo | latencia de red por request |

Ideal para healthcare/finanzas (privacidad) y demos. No para modelos > 100 MB ni datasets
propietarios que no querés exponer.

## 6. Generar la página demo (HTML + TF.js) a disco

In [ ]:
demo_html = '''<!doctype html>
<html><head><meta charset="utf-8">
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script></head>
<body>
<canvas id="c" width="28" height="28"></canvas>
<button id="predict">Predict</button>
<pre id="out"></pre>
<script>
let model;
tf.loadLayersModel('tfjs_model/model.json').then(m => model = m);
document.getElementById('predict').onclick = async () => {
  const ctx = document.getElementById('c').getContext('2d');
  const img = ctx.getImageData(0, 0, 28, 28).data;
  const gray = [];
  for (let i = 0; i < img.length; i += 4) gray.push(img[i] / 255.0);
  const probs = model.predict(tf.tensor2d(gray, [1, 784]));
  const clase = (await probs.argMax(1).data())[0];
  document.getElementById('out').textContent = 'clase: ' + clase;
};
</script></body></html>'''
with open("index.html", "w", encoding="utf-8") as f:
    f.write(demo_html)
print(f"index.html escrito ({len(demo_html)} chars) -> servir con: npx serve")

## 7. Snippet de detección de backend (WebGPU con fallback)

In [ ]:
backend_js = '''export async function initBackend() {
  try { await tf.setBackend('webgpu'); await tf.ready(); }
  catch { await tf.setBackend('webgl'); }        // fallback si no hay WebGPU
  return tf.getBackend();                          // 'webgpu' | 'webgl' | 'wasm'
}'''
with open("backend.js", "w", encoding="utf-8") as f:
    f.write(backend_js)
print("backend.js escrito: intenta WebGPU y cae a WebGL")

## 8. Validar la estructura de `model.json`

In [ ]:
import os, json
if os.path.exists("tfjs_model/model.json"):
    meta = json.load(open("tfjs_model/model.json", encoding="utf-8"))
    print("topologia:", "modelTopology" in meta)
    print("manifest de pesos:", "weightsManifest" in meta)
else:
    print("tfjs_model/model.json no existe todavia (correr el converter primero)")
    print("estructura esperada: modelTopology + weightsManifest (-> *.bin)")

## Ejercicios

1. Convertir un modelo Fashion-MNIST a TF.js e inspeccionar `model.json` y los `.bin`.
2. Armar una página HTML con un canvas donde el usuario dibuja y un botón "Predict".
3. Comparar la velocidad de inferencia con backend `webgl` vs `webgpu`.
4. Correr `sentiment-analysis` con transformers.js en el navegador y mostrar label + score.

## Conclusiones

- TF.js ejecuta modelos en el navegador: privacidad, sin server, sin latencia de red.
- `tensorflowjs_converter` produce `model.json` + pesos; en JS se carga con `tf.loadLayersModel`.
- El preprocesamiento en JS debe ser idéntico al de entrenamiento.
- WebGPU acelera 2-5× sobre WebGL; transformers.js trae modelos HF al browser vía ONNX.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Las de **núcleo numérico** son ejecutables (con `assert` de verificación); las de frameworks/servicios no instalados aquí (TF, PyTorch, diffusers, Gymnasium, GCP…) se muestran como **código real de referencia** listo para copiar en un entorno con esas dependencias.

### Ejercicio 1 — Convertir el modelo a TF.js

```bash
pip install tensorflowjs
tensorflowjs_converter --input_format=tf_saved_model servable/1/ tfjs_model/
# Genera model.json (topologia) + group1-shard*.bin (pesos)
```

### Ejercicio 2 — Página HTML que carga el modelo y predice

```html
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
<canvas id="c" width="28" height="28"></canvas>
<script type="module">
  const model = await tf.loadGraphModel('tfjs_model/model.json');
  const ctx = document.getElementById('c').getContext('2d');
  const img = tf.browser.fromPixels(ctx.getImageData(0, 0, 28, 28), 1)
                 .reshape([1, 784]).toFloat().div(255);
  const pred = model.predict(img);
  console.log('clase:', (await pred.argMax(1).data())[0]);
</script>
```

### Ejercicio 3 — Backend WebGPU

```javascript
import '@tensorflow/tfjs-backend-webgpu';
await tf.setBackend('webgpu');   // vs 'webgl'; WebGPU suele ser mas rapido
await tf.ready();
```

### Ejercicio 4 — transformers.js (modelo en el navegador, 1 línea)

```javascript
import { pipeline } from '@xenova/transformers';
const clf = await pipeline('sentiment-analysis');
console.log(await clf('I love running models in the browser!'));
// [{ label: 'POSITIVE', score: 0.999 }]  -- todo client-side, sin servidor
```

### Ejercicio 5 — Empaquetar como PWA (offline)

```javascript
// service-worker.js: cachea model.json + shards para inferencia offline
self.addEventListener('install', e => e.waitUntil(
  caches.open('v1').then(c => c.addAll(['/', '/tfjs_model/model.json',
                                        '/tfjs_model/group1-shard1of1.bin']))));
self.addEventListener('fetch', e =>
  e.respondWith(caches.match(e.request).then(r => r || fetch(e.request))));
```

> **Nota:** el converter es Python; el resto corre en el navegador (JS). No hay dependencias Python ejecutables aquí, por eso las soluciones son código real de referencia.